# Use case — `Utility/Pretraitement.py`

Clean Gaia light curves with time binning, adaptive robust outlier detection and source-level SNR rules. All hyperparameters are explained in the parameter cell.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT)) if str(PROJECT_ROOT) not in sys.path else None

from Utility.Pretraitement import pretraitement

In [ ]:
INPUT_CSV = PROJECT_ROOT / "data" / "raw_lightcurves.csv"       # Raw canonical Gaia table.
OUTPUT_DIR = PROJECT_ROOT / "results" / "preprocessing"          # Receives clean data and audit reports.

MAX_SOURCES = None             # None processes all groups; set a small integer for a quick test.
#when MAX_SOURCES is not None, then SOURCE_SELECTION and RANDOM_STATE enable random selection of MAX_SOURCES sources from the file
SOURCE_SELECTION = "first"     # "first" is deterministic; "random" samples groups using RANDOM_STATE.
RANDOM_STATE = 42              # Reproducible group selection when SOURCE_SELECTION="random".

DO_BINNING = True              # Merge nearby measurements before outlier detection.
BIN_DAYS = 4.0                 # Width, in days, of each within-component time bin.
ERROR_MODE = "mean"            # Aggregation rule for flux errors inside a bin.

HALF_WINDOW = 5                # Points on each side used by the rolling robust reference.
ALPHA = 0.7                    # Weight of MAD versus clipped-L2 scale in the hybrid dispersion.
CLIP_C = 2.5                   # Clipping constant used in the robust L2 contribution.
EXCLUDE_CENTER = True          # Excludes the tested point from its own local reference window.
SIGMA_THRESHOLD = 4.0          # Rejects points above this local robust-z magnitude.

DO_SNR_FILTER = True           # Enables source/component removal based on signal-to-noise.
SNR_SIGNAL_MODE = "amplitude"  # Robust signal definition used in the SNR denominator/numerator.
MIN_SNR = 0.8                  # Removes groups below this robust signal-to-noise ratio.
MAX_NOISE_SIGNAL_RATIO = None  # Optional upper noise/signal limit; None disables this second rule.
REMOVE_IF_SNR_NAN = False      # False keeps groups whose SNR cannot be evaluated.

MIN_POINTS_BEFORE = 3          # Minimum binned points needed before local cleaning.
MIN_POINTS_AFTER = 20          # Minimum surviving points required for a usable component.
MAX_OUTLIER_FRACTION = 0.50    # Removes a group if more than this fraction is flagged.

if not INPUT_CSV.exists():
    raise FileNotFoundError(f"Supply the raw canonical CSV first: {INPUT_CSV}")

In [ ]:
result = pretraitement(
    INPUT_CSV,
    output_dir=OUTPUT_DIR,
    max_sources=MAX_SOURCES,
    source_selection=SOURCE_SELECTION,
    random_state=RANDOM_STATE,
    do_binning=DO_BINNING,
    bin_days=BIN_DAYS,
    error_mode=ERROR_MODE,
    half_window=HALF_WINDOW,
    alpha=ALPHA,
    c=CLIP_C,
    exclude_center=EXCLUDE_CENTER,
    sigma_threshold=SIGMA_THRESHOLD,
    do_snr_filter=DO_SNR_FILTER,
    snr_signal_mode=SNR_SIGNAL_MODE,
    min_snr=MIN_SNR,
    max_noise_signal_ratio=MAX_NOISE_SIGNAL_RATIO,
    remove_if_snr_nan=REMOVE_IF_SNR_NAN,
    min_points_before=MIN_POINTS_BEFORE,
    min_points_after=MIN_POINTS_AFTER,
    max_outlier_fraction=MAX_OUTLIER_FRACTION,
    save=True,
    verbose=True,
)

display(result.source_report.head())
print(result.summary)
print("Clean CSV:", result.clean_csv_path)